---
title: "Machine Learning: Neural Networks as a Bridge to Deep Learning"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
    code-fold: true
jupyter: python
---


<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/12-neural-networks-bridge.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Neural Networks: A Bridge to Deep Learning**

Classical machine learning often separates **feature construction** from **prediction**. A practitioner designs a representation $\phi(\mathbf{x})$, then fits a linear model, kernel machine, tree ensemble, or probabilistic model on top. A neural network moves part of that representation design inside the optimization problem:

$$
f_{\theta}(\mathbf{x})
=f^{(L)}_{\theta_L}\circ f^{(L-1)}_{\theta_{L-1}}
\circ\cdots\circ f^{(1)}_{\theta_1}(\mathbf{x}).
$$

Each layer transforms the representation produced by the preceding layer. The final layer can still be familiar: linear regression for a continuous target, logistic or softmax regression for classification, or a score used by a structured decoder. What changes is that the features supplied to that head are learned jointly with it.

This chapter develops the minimum neural-network machinery needed to connect the earlier machine-learning chapters to the separate Deep Learning series. It focuses on concepts shared by nearly every architecture:

- an affine transformation followed by a nonlinear activation;
- composition into a multilayer function;
- a forward pass that maps a mini-batch to predictions and a scalar objective;
- backpropagation and reverse-mode automatic differentiation;
- initialization, optimization, and regularization;
- representation learning and parameter sharing.

It does **not** attempt to teach every neural architecture here. CNNs, RNNs, attention, and Transformers appear only as a map of their inductive biases. Their internal mechanisms, modern training practices, and applications belong to the Deep Learning guideline.

A neural network is not automatically “deep,” and “neural” does not imply biological realism. In machine learning, it is best understood as a parameterized computational graph made from differentiable operations. A one-hidden-layer multilayer perceptron is already a neural network; depth refers to the number of successive learned transformations through which information passes.

The four objects below should remain separate:

| Object | Role | Example |
|---|---|---|
| Architecture | Defines the function family and parameter sharing | MLP, CNN, Transformer |
| Forward computation | Produces intermediate activations and predictions | $\mathbf H=\operatorname{ReLU}(\mathbf X\mathbf W+\mathbf b)$ |
| Objective | States what training should improve | Cross-entropy plus weight decay |
| Optimizer | Uses gradients to change parameters | SGD, momentum, AdamW |

Backpropagation does not choose the model and does not perform the parameter update. It computes derivatives of the chosen scalar objective through the chosen computation graph. The optimizer then decides how to use those derivatives. This distinction prevents many conceptual errors later in deep learning.


### **From Linear Models to Artificial Neurons**

Linear and generalized linear models predict from a weighted sum

$$
z=\mathbf w^\top\mathbf x+b.
$$

An **artificial neuron** applies an activation function $g$ to this pre-activation:

$$
a=g(z)=g(\mathbf w^\top\mathbf x+b).
$$

This notation contains two distinct parts. The weights and bias learn a direction and threshold in feature space; the activation determines how the scalar pre-activation is transformed before it is passed onward. A network layer evaluates many such units at once with matrix multiplication.

#### **The Perceptron**

The classical perceptron is a linear binary classifier. For labels $y_i\in\{-1,+1\}$, it predicts

$$
\widehat y=\operatorname{sign}(\mathbf w^\top\mathbf x+b).
$$

Whenever $y_i(\mathbf w^\top\mathbf x_i+b)\leq 0$, the perceptron update is

$$
\mathbf w\leftarrow\mathbf w+\eta y_i\mathbf x_i,
\qquad
b\leftarrow b+\eta y_i.
$$

The update moves the decision hyperplane toward classifying the current example correctly. If the training data are linearly separable, the perceptron convergence theorem guarantees that repeated updates eventually find a separating hyperplane. It does not guarantee calibrated probabilities, a maximum-margin separator, or convergence on non-separable data.

<div class="diagram-scroll">

![A weighted artificial neuron and the geometric reason one perceptron cannot represent XOR.](assets/neuron-xor-boundary.svg){fig-alt="A perceptron computes a weighted sum followed by an activation and can represent only one linear decision boundary, which cannot separate XOR."}

</div>

<details>
<summary><strong>Python: train a perceptron on separable data</strong></summary>

```python
import numpy as np

X = np.array([
    [-2.0, -1.0], [-1.0, -1.5], [-1.5, -0.5],
    [1.0, 1.5], [2.0, 1.0], [1.5, 2.0],
])
y = np.array([-1, -1, -1, 1, 1, 1])

weights = np.zeros(X.shape[1])
bias = 0.0
learning_rate = 0.5

for epoch in range(20):
    mistakes = 0
    for features, target in zip(X, y):
        margin = target * (features @ weights + bias)
        if margin <= 0:
            # Move the hyperplane in the direction that favors this target.
            weights += learning_rate * target * features
            bias += learning_rate * target
            mistakes += 1
    if mistakes == 0:
        break

predictions = np.where(X @ weights + bias >= 0, 1, -1)
print("epochs used:", epoch + 1)
print("weights, bias:", np.round(weights, 3), round(bias, 3))
print("training predictions:", predictions.tolist())
```

</details>

The XOR truth table exposes the limitation. The positive points occupy opposite corners and cannot be separated from the negative points by one line. A perceptron can continue updating forever without finding a zero-error solution because the required separator does not exist in the original feature space.

<details>
<summary><strong>Python: observe perceptron cycling on XOR</strong></summary>

```python
import numpy as np

X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = np.array([-1, 1, 1, -1])
weights = np.zeros(2)
bias = 0.0
mistakes_by_epoch = []

for _ in range(25):
    mistakes = 0
    for features, target in zip(X, y):
        if target * (features @ weights + bias) <= 0:
            weights += target * features
            bias += target
            mistakes += 1
    mistakes_by_epoch.append(mistakes)

predictions = np.where(X @ weights + bias >= 0, 1, -1)
print("last five mistake counts:", mistakes_by_epoch[-5:])
print("final predictions:", predictions.tolist())
print("perfect separation:", bool(np.all(predictions == y)))
```

</details>

This failure does not mean gradient-based learning is impossible. It means the hypothesis class is too restrictive. One response is manual feature engineering, such as adding $x_1x_2$. A neural network instead learns intermediate nonlinear features that can make the final task linearly separable.

#### **Linear Layers and Activation Functions**

For a mini-batch $\mathbf X\in\mathbb R^{B\times d}$ and a layer with $h$ units,

$$
\mathbf Z=\mathbf X\mathbf W+\mathbf b,
\qquad
\mathbf W\in\mathbb R^{d\times h},
\quad
\mathbf b\in\mathbb R^{h}.
$$

The bias is broadcast across the $B$ examples, and an elementwise activation produces $\mathbf H=g(\mathbf Z)$. Without $g$, stacking affine layers adds no expressive power:

$$
(\mathbf X\mathbf W_1+\mathbf b_1)\mathbf W_2+\mathbf b_2
=\mathbf X(\mathbf W_1\mathbf W_2)
+(\mathbf b_1\mathbf W_2+\mathbf b_2),
$$

which is still one affine transformation. Nonlinearity is therefore the ingredient that prevents the layers from collapsing into a single linear model.

<div class="diagram-scroll">

![Sigmoid, tanh, ReLU, and GELU-style activation functions with their ranges and gradient behavior.](assets/activation-functions-map.svg){fig-alt="Activation functions compared by shape, output range, saturation, and gradient behavior."}

</div>

Common choices serve different roles:

| Activation | Definition | Typical role | Main caution |
|---|---|---|---|
| Sigmoid | $\sigma(z)=1/(1+e^{-z})$ | Binary probability output, gates | Hidden gradients vanish in saturated tails |
| Tanh | $\tanh(z)$ | Centered bounded state, recurrent gates | Also saturates for large $|z|$ |
| ReLU | $\max(0,z)$ | Default hidden activation in many MLP/CNN settings | Units can remain inactive on the negative side |
| Leaky ReLU | $\max(\alpha z,z)$ | Preserve a small negative gradient | Extra slope choice |
| GELU | approximately $z\Phi(z)$ | Smooth gating in Transformers | More computation and no simple piecewise-linear form |

The hidden activation and output link should be chosen separately. ReLU can be appropriate inside a binary classifier whose final logit is passed through sigmoid. Softmax belongs at a mutually exclusive multiclass output, not necessarily in hidden layers. For numerical stability, training code usually passes raw logits directly to a combined cross-entropy implementation instead of explicitly computing probabilities first.

<details>
<summary><strong>Python: compare activation values and derivatives</strong></summary>

```python
import numpy as np

z = np.array([-6.0, -2.0, 0.0, 2.0, 6.0])
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)
relu = np.maximum(0.0, z)
leaky_relu = np.where(z >= 0, z, 0.05 * z)

derivatives = {
    "sigmoid": sigmoid * (1 - sigmoid),
    "tanh": 1 - tanh**2,
    "relu": (z > 0).astype(float),
    "leaky_relu": np.where(z >= 0, 1.0, 0.05),
}

print("z:", z)
print("sigmoid:", np.round(sigmoid, 4))
print("tanh:", np.round(tanh, 4))
print("relu:", relu)
for name, derivative in derivatives.items():
    print(f"{name:11s} derivative:", np.round(derivative, 4))
```

</details>

The derivative table explains saturation directly: sigmoid derivatives approach zero in both tails, so multiplying many such derivatives through depth can erase a gradient. ReLU avoids positive-side saturation, although it introduces a zero-gradient negative half-space. Activation choice is therefore about optimization as well as expressivity.


### **Multilayer Perceptrons**

A **multilayer perceptron (MLP)**, also called a feedforward fully connected network, composes affine layers and nonlinear activations without cycles. For one hidden layer,

$$
\mathbf h=g(\mathbf W_1^\top\mathbf x+\mathbf b_1),
\qquad
\mathbf o=\mathbf W_2^\top\mathbf h+\mathbf b_2.
$$

The input layer is a placeholder for features and is not normally counted as a learned layer. The **width** is the number of units in a hidden layer; the **depth** is the number of successive parameterized transformations. A model can be wide but shallow, deep but narrow, or both.

#### **Hidden Layers and Representation Capacity**

Each hidden unit introduces a learned feature. With ReLU, one unit changes behavior across the hyperplane $\mathbf w^\top\mathbf x+b=0$. Combining many units divides input space into regions and assigns a different affine function to each region. Adding depth composes these partitions, allowing complicated decision boundaries to be represented compactly.

<div class="diagram-scroll">

![An MLP viewed as successive learned coordinate systems ending in a simple linear output head.](assets/mlp-composition-regions.svg){fig-alt="An MLP transforms raw inputs through nonlinear hidden representations until a linear output boundary is sufficient."}

</div>

XOR becomes simple after two hidden features. Let

$$
h_1=\operatorname{ReLU}(x_1+x_2),
\qquad
h_2=\operatorname{ReLU}(x_1+x_2-1).
$$

For binary inputs, $h_1$ detects whether at least one input is active, while $h_2$ detects whether both are active. The output

$$
o=h_1-2h_2
$$

is exactly the XOR value. The final head is linear; the hidden representation made linear prediction possible.

<details>
<summary><strong>Python: represent XOR with a fixed ReLU hidden layer</strong></summary>

```python
import numpy as np

X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])

# Two hidden units: h1 detects "at least one" and h2 detects "both".
W1 = np.array([[1.0, 1.0], [1.0, 1.0]])
b1 = np.array([0.0, -1.0])
hidden = np.maximum(0.0, X @ W1 + b1)

W2 = np.array([1.0, -2.0])
output = hidden @ W2

print("hidden representation:\n", hidden)
print("XOR output:", output.astype(int).tolist())
```

</details>

This construction proves representational possibility, not learnability. Training must discover useful hidden units from finite data using a non-convex objective. Width and depth increase capacity, but they also increase parameter count, computation, and the number of equivalent or poorly conditioned parameter configurations.

Parameter count for a dense layer from $d$ inputs to $h$ outputs is

$$
dh+h,
$$

including weights and biases. Fully connected layers therefore become expensive for raw images, long sequences, or very high-dimensional inputs. CNNs, recurrent models, and attention architectures introduce structured parameter sharing rather than connecting every input dimension independently to every unit.

#### **Universal Approximation Intuition**

Universal approximation results state, under suitable conditions, that a network with at least one sufficiently wide hidden layer and an appropriate nonlinearity can approximate any continuous function on a compact domain to arbitrary accuracy. The theorem is an **existence statement**:

- it does not say how many units are required;
- it does not say gradient descent will find the required parameters;
- it does not guarantee good generalization from finite samples;
- it does not imply a shallow network is computationally efficient.

For one-dimensional inputs, ReLU networks provide concrete intuition. A continuous piecewise-linear function can be written as a base line plus weighted hinges:

$$
f(x)=a+bx+\sum_{j=1}^{m}c_j\operatorname{ReLU}(x-t_j).
$$

Each hinge changes the slope after knot $t_j$. With enough knots, linear interpolation approximates a smooth target increasingly well.

<details>
<summary><strong>Python: approximate a smooth function with ReLU hinges</strong></summary>

```python
import numpy as np

def relu_spline_parameters(knots, values):
    """Represent linear interpolation using a base line plus ReLU hinges."""
    slopes = np.diff(values) / np.diff(knots)
    base_intercept = values[0] - slopes[0] * knots[0]
    slope_changes = np.diff(slopes)
    return base_intercept, slopes[0], slope_changes

def evaluate_relu_spline(x, knots, values):
    intercept, base_slope, slope_changes = relu_spline_parameters(knots, values)
    prediction = intercept + base_slope * x
    for knot, change in zip(knots[1:-1], slope_changes):
        prediction += change * np.maximum(0.0, x - knot)
    return prediction

x = np.linspace(-np.pi, np.pi, 500)
for n_knots in (5, 9, 17):
    knots = np.linspace(-np.pi, np.pi, n_knots)
    values = np.sin(knots)
    prediction = evaluate_relu_spline(x, knots, values)
    max_error = np.max(np.abs(prediction - np.sin(x)))
    print(f"knots={n_knots:2d}, max error={max_error:.5f}")
```

</details>

The approximation improves as width increases because each added ReLU supplies another possible change in slope. Depth can reuse and compose intermediate features, representing some functions with far fewer units than a single very wide layer. Which functions gain such efficiency depends on the compositional structure of the problem.

The theorem also explains why capacity alone is an incomplete model-selection argument. A lookup table can represent any labels on a finite training set, yet may generalize poorly. The practical question is whether the architecture, optimization procedure, regularization, and data jointly favor a useful function among all representable functions.


### **The Forward Pass and Objective**

The **forward pass** evaluates the network from input to output for a mini-batch. It computes both the final prediction and intermediate values that backward propagation will need. For a one-hidden-layer classifier with $B$ examples, $d$ features, $h$ hidden units, and $K$ classes,

$$
\mathbf X\in\mathbb R^{B\times d},
\quad
\mathbf W_1\in\mathbb R^{d\times h},
\quad
\mathbf W_2\in\mathbb R^{h\times K},
$$

$$
\mathbf Z_1=\mathbf X\mathbf W_1+\mathbf b_1,
\qquad
\mathbf H=\operatorname{ReLU}(\mathbf Z_1),
$$

$$
\mathbf O=\mathbf H\mathbf W_2+\mathbf b_2.
$$

$\mathbf O$ contains **logits**, unconstrained scores before the output distribution or task loss. Tracking dimensions is one of the most effective ways to prevent implementation mistakes: every matrix multiplication must agree on its inner dimensions, and every bias must broadcast over the intended axis.

<div class="diagram-scroll">

![A mini-batch moving through affine and activation layers to logits, data loss, regularization, and one scalar objective.](assets/forward-pass-objective.svg){fig-alt="The forward pass with tensor shapes, intermediate activations, an output head, and the final regularized objective."}

</div>

The output head and loss encode the task:

| Task | Network output | Common training loss | Prediction |
|---|---|---|---|
| Regression | $q$ unrestricted values | MSE, MAE, Gaussian NLL | Direct value or distribution parameters |
| Binary classification | One logit $o$ | Binary cross-entropy with logits | $\sigma(o)$ then threshold or cost rule |
| Multiclass classification | $K$ logits | Softmax cross-entropy | $\arg\max_k o_k$ |
| Multilabel classification | One logit per label | Independent binary cross-entropies | Per-label probabilities and thresholds |

For one multiclass example with target class $y$, softmax probability and cross-entropy are

$$
p_k=\frac{e^{o_k}}{\sum_j e^{o_j}},
\qquad
\ell=-\log p_y.
$$

Subtracting $m=\max_j o_j$ from every logit leaves the probabilities unchanged but prevents overflow:

$$
\log\sum_j e^{o_j}
=m+\log\sum_j e^{o_j-m}.
$$

The stable loss can therefore be computed directly from logits as

$$
\ell=-o_y+m+\log\sum_j e^{o_j-m}.
$$

<details>
<summary><strong>Python: execute a shape-safe forward pass and stable cross-entropy</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(5)
batch_size, n_features, n_hidden, n_classes = 4, 3, 5, 3
X = rng.normal(size=(batch_size, n_features))
y = np.array([0, 2, 1, 2])

W1 = rng.normal(scale=np.sqrt(2 / n_features), size=(n_features, n_hidden))
b1 = np.zeros(n_hidden)
W2 = rng.normal(scale=np.sqrt(2 / n_hidden), size=(n_hidden, n_classes))
b2 = np.zeros(n_classes)

Z1 = X @ W1 + b1
H = np.maximum(0.0, Z1)
logits = H @ W2 + b2

shifted = logits - logits.max(axis=1, keepdims=True)
log_normalizer = np.log(np.exp(shifted).sum(axis=1))
losses = -shifted[np.arange(batch_size), y] + log_normalizer

print("X, H, logits shapes:", X.shape, H.shape, logits.shape)
print("per-example losses:", np.round(losses, 4))
print("mean loss:", round(losses.mean(), 4))
```

</details>

Training minimizes a scalar objective, commonly

$$
J(\theta)
=\frac{1}{B}\sum_{i=1}^{B}
\ell(f_\theta(\mathbf x_i),y_i)
+\lambda R(\theta).
$$

The first term measures mini-batch fit; $R(\theta)$ may be an $L_2$ penalty or another regularizer. The choice between summing and averaging the batch loss changes gradient scale and therefore interacts with learning rate. A well-defined implementation must also specify whether regularization includes biases and normalization parameters.

The forward pass behaves differently in training and evaluation modes. Dropout is stochastic during training but disabled or analytically rescaled during evaluation. Batch-dependent normalization layers update and use statistics differently. Validation must run in evaluation mode and without parameter updates; otherwise the measured model is not the one that will be deployed.

<details>
<summary><strong>Python: use task-appropriate output transformations</strong></summary>

```python
import numpy as np

raw_output = np.array([
    [2.0, -1.0, 0.5],
    [-0.5, 1.2, 0.3],
])

# Mutually exclusive classes: one softmax distribution per row.
shifted = raw_output - raw_output.max(axis=1, keepdims=True)
softmax = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)

# Independent labels: one sigmoid probability per output dimension.
sigmoid = 1 / (1 + np.exp(-raw_output))

# Regression: raw outputs may be used directly.
regression = raw_output[:, :1]

print("softmax row sums:", softmax.sum(axis=1))
print("multiclass probabilities:\n", np.round(softmax, 3))
print("multilabel probabilities:\n", np.round(sigmoid, 3))
print("regression predictions:", regression.ravel())
```

</details>

Softmax outputs compete and sum to one; sigmoid outputs do not. Using softmax for a multilabel task incorrectly forces the presence of one label to suppress every other label. The architecture of the output layer is therefore part of the statistical formulation, not a cosmetic final step.


### **Backpropagation**

**Backpropagation** efficiently computes the derivative of one scalar objective with respect to every parameter in a network. It is reverse-mode differentiation applied to a layered computational graph. The algorithm consists of ordinary calculus plus two organizational ideas:

1. cache intermediate forward values instead of recomputing every subexpression;
2. propagate an upstream derivative backward through each local operation, adding contributions when a value influences the loss through multiple paths.

Backpropagation answers “How would an infinitesimal change in this parameter change the current objective?” It does not decide the learning rate, regularization strength, or update rule, and it does not guarantee that the resulting optimization problem is convex.

#### **Computational Graphs**

A computational graph represents values as nodes and primitive operations as edges or operation nodes. Consider

$$
a=wx,
\qquad
b=a+a,
\qquad
c=\tanh(b),
\qquad
L=(c-y)^2.
$$

The forward pass evaluates $a,b,c,L$. The reverse pass starts with $\partial L/\partial L=1$ and applies local derivatives in reverse topological order. Because $a$ is used twice in $b=a+a$, its two gradient contributions add. This accumulation rule is essential for residual connections, parameter sharing, and recurrent networks.

<div class="diagram-scroll">

![A two-layer network with forward values moving toward the loss and gradients returning to every parameter.](assets/backprop-computational-graph.svg){fig-alt="Backpropagation traverses the cached forward computation in reverse and accumulates vector-Jacobian products."}

</div>

Every node receives an **upstream gradient** $\bar v=\partial L/\partial v$. For an operation $v=f(u)$, the backward rule computes

$$
\bar u
+=\bar v\frac{\partial v}{\partial u}.
$$

The `+=` matters: $u$ may feed several downstream operations. In vector code, frameworks do not usually construct a full Jacobian. They compute a **vector-Jacobian product (VJP)**, multiplying the upstream gradient by the local Jacobian in the order needed by reverse mode.

#### **The Chain Rule through Layers**

For the two-layer network

$$
\mathbf Z_1=\mathbf X\mathbf W_1+\mathbf b_1,
\quad
\mathbf H=g(\mathbf Z_1),
\quad
\mathbf O=\mathbf H\mathbf W_2+\mathbf b_2,
\quad
L=\ell(\mathbf O,\mathbf y),
$$

the chain rule passes sensitivity through the composition:

$$
\frac{\partial L}{\partial \mathbf Z_1}
=
\frac{\partial L}{\partial \mathbf O}
\frac{\partial \mathbf O}{\partial \mathbf H}
\odot g'(\mathbf Z_1).
$$

$\odot$ denotes elementwise multiplication. For ReLU, $g'(z)=1$ when $z>0$ and $0$ when $z<0$; a conventional subgradient is chosen at $z=0$. The activation mask therefore gates which hidden units receive a gradient for the current mini-batch.

For softmax cross-entropy, a useful simplification occurs. If $\mathbf p_i=\operatorname{softmax}(\mathbf o_i)$ and $\mathbf e_{y_i}$ is the one-hot target, then for the mean batch loss

$$
\frac{\partial L}{\partial \mathbf o_i}
=\frac{1}{B}(\mathbf p_i-\mathbf e_{y_i}).
$$

The gradient increases the target logit relative to the others. It is a gradient with respect to logits, not probabilities, which is another reason stable libraries combine softmax and cross-entropy.

#### **Parameter Gradients**

Let $\mathbf G_O=\partial L/\partial\mathbf O$. Matrix calculus gives

$$
\frac{\partial L}{\partial\mathbf W_2}
=\mathbf H^\top\mathbf G_O,
\qquad
\frac{\partial L}{\partial\mathbf b_2}
=\sum_{i=1}^{B}\mathbf G_{O,i},
$$

$$
\mathbf G_H=\mathbf G_O\mathbf W_2^\top,
\qquad
\mathbf G_{Z_1}=\mathbf G_H\odot g'(\mathbf Z_1),
$$

$$
\frac{\partial L}{\partial\mathbf W_1}
=\mathbf X^\top\mathbf G_{Z_1},
\qquad
\frac{\partial L}{\partial\mathbf b_1}
=\sum_{i=1}^{B}\mathbf G_{Z_1,i}.
$$

These shapes match their parameters. The transpose in $\mathbf H^\top\mathbf G_O$ aggregates evidence across the batch and forms every hidden-to-output weight gradient. Bias gradients sum over examples because the same bias is shared across the batch.

<details>
<summary><strong>Python: implement a complete two-layer backward pass</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(8)
B, d, h, K = 5, 4, 6, 3
X = rng.normal(size=(B, d))
y = np.array([0, 2, 1, 1, 0])
W1 = rng.normal(scale=0.3, size=(d, h))
b1 = np.zeros(h)
W2 = rng.normal(scale=0.3, size=(h, K))
b2 = np.zeros(K)

# Forward pass: cache Z1, H, and probabilities for backward use.
Z1 = X @ W1 + b1
H = np.maximum(0.0, Z1)
logits = H @ W2 + b2
shifted = logits - logits.max(axis=1, keepdims=True)
probabilities = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
loss = -np.log(probabilities[np.arange(B), y]).mean()

# Backward pass begins at the mean softmax cross-entropy.
d_logits = probabilities.copy()
d_logits[np.arange(B), y] -= 1
d_logits /= B

dW2 = H.T @ d_logits
db2 = d_logits.sum(axis=0)
dH = d_logits @ W2.T
dZ1 = dH * (Z1 > 0)
dW1 = X.T @ dZ1
db1 = dZ1.sum(axis=0)

print("loss:", round(loss, 4))
print("gradient shapes:", dW1.shape, db1.shape, dW2.shape, db2.shape)
print("gradient norms:", round(np.linalg.norm(dW1), 4),
      round(np.linalg.norm(dW2), 4))
```

</details>

The implementation makes the forward/backward dependency explicit. If an intermediate activation is discarded, it must be recomputed or recovered. Modern systems trade memory for computation using checkpointing: retain selected activations and recompute others during backward propagation.

Analytical gradients should be tested on small deterministic problems with **finite differences**. For parameter coordinate $\theta_j$,

$$
\frac{\partial L}{\partial\theta_j}
\approx
\frac{L(\theta_j+\varepsilon)-L(\theta_j-\varepsilon)}{2\varepsilon}.
$$

The centered difference has lower truncation error than a one-sided difference. $\varepsilon$ must be small but not so small that floating-point cancellation dominates. Gradient checking is a debugging tool, not a training method: it requires two forward passes per checked parameter.

<details>
<summary><strong>Python: verify one analytical gradient with finite differences</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(2)
X = rng.normal(size=(4, 3))
y = np.array([0, 1, 1, 0])
W1 = rng.normal(scale=0.4, size=(3, 5))
b1 = rng.normal(scale=0.1, size=5)
W2 = rng.normal(scale=0.4, size=(5, 2))
b2 = rng.normal(scale=0.1, size=2)

def loss_and_dW1(W1_value):
    Z1 = X @ W1_value + b1
    H = np.tanh(Z1)
    logits = H @ W2 + b2
    shifted = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    loss = -np.log(probabilities[np.arange(len(X)), y]).mean()

    d_logits = probabilities
    d_logits[np.arange(len(X)), y] -= 1
    d_logits /= len(X)
    dH = d_logits @ W2.T
    dZ1 = dH * (1 - H**2)
    return loss, X.T @ dZ1

_, analytical = loss_and_dW1(W1)
row, column = 1, 3
epsilon = 1e-5
W_plus = W1.copy(); W_plus[row, column] += epsilon
W_minus = W1.copy(); W_minus[row, column] -= epsilon
loss_plus, _ = loss_and_dW1(W_plus)
loss_minus, _ = loss_and_dW1(W_minus)
numerical = (loss_plus - loss_minus) / (2 * epsilon)

relative_error = abs(analytical[row, column] - numerical) / max(
    1e-12, abs(analytical[row, column]) + abs(numerical)
)
print("analytical:", analytical[row, column])
print("numerical :", numerical)
print("relative error:", relative_error)
```

</details>

Gradient checks can fail near non-differentiable points such as a ReLU input of exactly zero, so smooth activations or carefully chosen test inputs are useful. Passing a numerical check strongly supports a local implementation, but it does not prove that the model objective or data pipeline is conceptually correct.


### **Automatic Differentiation**

Writing every backward equation by hand is educational but fragile. **Automatic differentiation (autodiff)** decomposes a program into primitive operations with known local derivative rules and applies the chain rule automatically to the values produced by one execution.

Autodiff differs from two alternatives:

- **symbolic differentiation** manipulates an algebraic expression and can create very large formulas with repeated subexpressions;
- **numerical differentiation** perturbs parameters and estimates slopes, which is slow and approximate;
- **automatic differentiation** evaluates exact derivatives of the implemented floating-point computation up to ordinary rounding error.

Two modes organize the chain rule differently. **Forward mode** propagates derivatives from selected inputs toward outputs and is efficient when there are few inputs and many outputs. **Reverse mode** propagates one scalar output's sensitivity back to many inputs and is ideal for neural training, where one loss depends on millions of parameters.

<div class="diagram-scroll">

![A forward program recorded on an automatic-differentiation tape and traversed in reverse to accumulate adjoints.](assets/reverse-mode-autodiff.svg){fig-alt="Reverse-mode automatic differentiation records primitive operations and cached values, then propagates adjoints in reverse topological order."}

</div>

Suppose $F:\mathbb R^n\rightarrow\mathbb R^m$ has Jacobian $J$. Forward mode efficiently computes a Jacobian-vector product

$$
J\mathbf v,
$$

while reverse mode computes a vector-Jacobian product

$$
\mathbf u^\top J.
$$

For a scalar loss, $m=1$ and $\mathbf u=1$, so one reverse pass returns the full gradient $\nabla_\theta L$. Frameworks store a dynamic or compiled graph, cached tensors needed by local backward rules, and an adjoint for each differentiable value.

<details>
<summary><strong>Python: build a tiny reverse-mode autodiff engine</strong></summary>

```python
import math

class Value:
    def __init__(self, data, parents=(), operation=""):
        self.data = float(data)
        self.grad = 0.0
        self.parents = tuple(parents)
        self.operation = operation
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = backward
        return out

    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = backward
        return out

    __rmul__ = __mul__

    def __sub__(self, other):
        return self + (-1.0 * other)

    def __pow__(self, power):
        out = Value(self.data**power, (self,), f"**{power}")
        def backward():
            self.grad += power * self.data ** (power - 1) * out.grad
        out._backward = backward
        return out

    def tanh(self):
        value = math.tanh(self.data)
        out = Value(value, (self,), "tanh")
        def backward():
            self.grad += (1 - value**2) * out.grad
        out._backward = backward
        return out

    def backward(self):
        order, visited = [], set()
        def build(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    build(parent)
                order.append(node)
        build(self)
        self.grad = 1.0
        for node in reversed(order):
            node._backward()

x = Value(1.5)
w = Value(-0.7)
y = Value(0.2)
a = x * w
b = a + a                 # the reused value a receives two gradient paths
c = b.tanh()
loss = (c - y) ** 2
loss.backward()

print("loss:", round(loss.data, 6))
print("dL/dx:", round(x.grad, 6))
print("dL/dw:", round(w.grad, 6))
print("gradient accumulated at reused a:", round(a.grad, 6))
```

</details>

The graph records actual operations executed, so control flow can be differentiated along the path that ran. A dynamic graph is intuitive and flexible; a compiled graph can enable operation fusion, memory planning, parallel scheduling, and hardware-specific optimization. Modern frameworks often trace or compile parts of dynamic programs to obtain both flexibility and performance.

Autodiff still requires discipline:

- detaching a tensor intentionally or accidentally breaks the gradient path;
- in-place mutation can overwrite a cached value needed in backward propagation;
- gradients usually accumulate until explicitly cleared;
- non-differentiable discrete choices need estimators, relaxations, or alternative learning methods;
- a differentiable program can still contain unstable operations or a poorly specified objective.

Autodiff removes derivative bookkeeping, not mathematical responsibility. Understanding local derivatives, tensor shapes, and expected gradient scale remains necessary for diagnosing silent errors.


### **Training Neural Networks**

Neural-network training combines a statistical objective with a numerical optimization process. A typical loop repeatedly samples a mini-batch, performs a forward pass, computes the scalar loss, backpropagates gradients, updates parameters, and periodically evaluates an untouched validation set. Each component affects the others: initialization changes gradient scale, batch size changes gradient noise, the optimizer changes effective step sizes, and regularization changes which solution is preferred.

#### **Initialization and Gradient Flow**

All units in one layer should not begin identically. If every hidden weight and bias is zero, every unit receives the same gradient and remains identical, so width is wasted. Random initialization breaks this symmetry, but its scale must preserve useful signals through depth.

For independent zero-mean inputs and weights,

$$
z_j=\sum_{i=1}^{n_{\mathrm{in}}}w_{ij}x_i
$$

has approximate variance

$$
\operatorname{Var}(z_j)
\approx n_{\mathrm{in}}
\operatorname{Var}(w_{ij})
\operatorname{Var}(x_i).
$$

If weight variance is too small, activations and gradients shrink with every layer. If it is too large, they grow or drive saturating activations into flat regions. Variance-aware schemes choose an initial scale based on fan-in and fan-out:

$$
\operatorname{Var}(W_{ij})
\approx \frac{2}{n_{\mathrm{in}}+n_{\mathrm{out}}}
\quad\text{(Xavier/Glorot, often for tanh-like layers)},
$$

$$
\operatorname{Var}(W_{ij})
\approx \frac{2}{n_{\mathrm{in}}}
\quad\text{(He/Kaiming, for ReLU-like layers)}.
$$

<div class="diagram-scroll">

![Activation and gradient scale vanishing, remaining stable, or exploding across depth under different initialization scales.](assets/gradient-flow-initialization.svg){fig-alt="Too-small, variance-aware, and too-large initialization compared by signal scale across layers."}

</div>

These formulas preserve variance only approximately. Independence assumptions break during training, activation distributions change, and architecture-specific components alter signal flow. Residual connections, normalization, gated units, and careful optimizers help, but initialization still determines whether the first updates begin in a usable regime.

<details>
<summary><strong>Python: inspect activation variance through a deep random network</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(10)
batch_size, width, depth = 512, 128, 20
initial_inputs = rng.normal(size=(batch_size, width))

def variance_trace(weight_scale, activation):
    hidden = initial_inputs.copy()
    trace = [hidden.var()]
    for _ in range(depth):
        weights = rng.normal(scale=weight_scale, size=(width, width))
        hidden = activation(hidden @ weights)
        trace.append(hidden.var())
    return np.array(trace)

traces = {
    "too small, tanh": variance_trace(0.02, np.tanh),
    "Xavier, tanh": variance_trace(np.sqrt(1 / width), np.tanh),
    "He, ReLU": variance_trace(np.sqrt(2 / width), lambda z: np.maximum(0, z)),
    "too large, tanh": variance_trace(0.30, np.tanh),
}

for name, trace in traces.items():
    print(f"{name:18s} first={trace[0]:.3f}, middle={trace[10]:.3f}, last={trace[-1]:.3f}")
```

</details>

Activation variance alone is not a complete gradient diagnostic, but a collapse to nearly constant activations or immediate saturation is an early warning. During real training, monitor layerwise activation statistics, gradient norms, the fraction of active ReLU units, and update-to-parameter ratios.

#### **Mini-Batches and Optimizers**

For empirical risk

$$
J(\theta)=\frac{1}{n}\sum_{i=1}^{n}\ell_i(\theta),
$$

a uniformly sampled mini-batch $\mathcal B$ provides the stochastic gradient

$$
\widehat{\mathbf g}
=\frac{1}{|\mathcal B|}\sum_{i\in\mathcal B}\nabla_\theta\ell_i(\theta).
$$

It is an unbiased estimator of the full gradient under ordinary random sampling. Smaller batches use less memory and produce noisier updates; larger batches improve hardware utilization and reduce gradient variance but may require learning-rate adjustment and can perform fewer parameter updates per pass through the data. One **epoch** means approximately one full traversal of the training examples, not one update.

Plain SGD updates

$$
\theta_{t+1}=\theta_t-\eta_t\widehat{\mathbf g}_t.
$$

Momentum accumulates a velocity that smooths oscillation and accelerates persistent directions:

$$
\mathbf v_t=\beta\mathbf v_{t-1}+\widehat{\mathbf g}_t,
\qquad
\theta_{t+1}=\theta_t-\eta_t\mathbf v_t.
$$

Adam tracks exponential averages of first and second gradient moments, applies bias correction, and rescales each coordinate. This can make early optimization forgiving, especially with sparse or differently scaled gradients. It still needs a learning-rate schedule and can generalize differently from momentum SGD.

AdamW **decouples weight decay** from the adaptive gradient update. Adding an $L_2$ penalty to the loss and applying multiplicative parameter decay are equivalent for plain SGD under a simple learning rate, but not for coordinate-adaptive methods such as Adam. Biases and normalization parameters are often excluded from decay.

<details>
<summary><strong>Python: compare optimizers on an ill-conditioned objective</strong></summary>

```python
import numpy as np

# f(theta) = 0.5 * theta^T A theta has a narrow curved-looking ravine
# when eigenvalues of A differ greatly.
A = np.diag([1.0, 80.0])
start = np.array([4.0, 4.0])

def gradient(theta):
    return A @ theta

def run_sgd(steps=120, learning_rate=0.02):
    theta = start.copy()
    for _ in range(steps):
        theta -= learning_rate * gradient(theta)
    return theta

def run_momentum(steps=120, learning_rate=0.015, beta=0.9):
    theta = start.copy(); velocity = np.zeros_like(theta)
    for _ in range(steps):
        velocity = beta * velocity + gradient(theta)
        theta -= learning_rate * velocity
    return theta

def run_adam(steps=120, learning_rate=0.12, beta1=0.9, beta2=0.999):
    theta = start.copy(); first = np.zeros_like(theta); second = np.zeros_like(theta)
    for step in range(1, steps + 1):
        grad = gradient(theta)
        first = beta1 * first + (1 - beta1) * grad
        second = beta2 * second + (1 - beta2) * grad**2
        first_hat = first / (1 - beta1**step)
        second_hat = second / (1 - beta2**step)
        theta -= learning_rate * first_hat / (np.sqrt(second_hat) + 1e-8)
    return theta

for name, result in {
    "SGD": run_sgd(),
    "momentum": run_momentum(),
    "Adam": run_adam(),
}.items():
    objective = 0.5 * result @ A @ result
    print(f"{name:8s} theta={np.round(result, 5)}, objective={objective:.8f}")
```

</details>

The example demonstrates optimization geometry, not a universal ranking. An optimizer that reaches the minimum fastest on one quadratic need not yield the best validation performance on a neural model. Optimizer comparisons must use tuned learning rates, comparable update budgets, and the same data order and evaluation protocol.

#### **Dropout, Weight Decay, and Early Stopping**

Regularization controls generalization rather than merely reducing training loss.

**Dropout** multiplies hidden activations by independent Bernoulli masks during training. With keep probability $q$ and inverted scaling,

$$
\widetilde{\mathbf h}
=\frac{\mathbf m\odot\mathbf h}{q},
\qquad
m_j\sim\operatorname{Bernoulli}(q).
$$

Then $\mathbb E[\widetilde{\mathbf h}]=\mathbf h$, so evaluation uses the unmasked activation directly. Dropout injects multiplicative noise and discourages fragile co-adaptation, but excessive dropout can cause underfitting and may be unnecessary in some normalized or very large pretrained models.

**Weight decay** shrinks selected weights toward zero and discourages overly large parameter norms. It can improve conditioning and generalization, but it does not directly impose sparsity. **Early stopping** treats the training trajectory as a sequence of candidate models: save the checkpoint with the best validation metric and stop after no meaningful improvement for a chosen patience. The final test set must not control this decision.

<div class="diagram-scroll">

![The mini-batch training loop with dropout, weight decay, optimizer updates, validation checkpoints, and early stopping.](assets/training-regularization-loop.svg){fig-alt="A neural training loop showing the separate roles of optimization, regularization, and validation-based early stopping."}

</div>

<details>
<summary><strong>Python: verify inverted-dropout expectation</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(11)
hidden = np.array([0.5, 1.0, 2.0, -1.0])
keep_probability = 0.7
samples = []

for _ in range(50_000):
    mask = rng.binomial(1, keep_probability, size=hidden.shape)
    samples.append(mask * hidden / keep_probability)

mean_training_activation = np.mean(samples, axis=0)
print("original evaluation activation:", hidden)
print("mean dropout activation       :", np.round(mean_training_activation, 3))
```

</details>

<details>
<summary><strong>Python: train a regularized MLP with validation-based stopping</strong></summary>

```python
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Nonlinear data makes a hidden representation useful.
X, y = make_moons(n_samples=1200, noise=0.22, random_state=4)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=4
)

model = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=1e-3,              # L2 regularization in this estimator
        batch_size=64,
        learning_rate_init=2e-3,
        early_stopping=True,     # reserve part of training data for validation
        validation_fraction=0.15,
        n_iter_no_change=20,
        max_iter=500,
        random_state=4,
    ),
)
model.fit(X_train, y_train)

probabilities = model.predict_proba(X_test)
predictions = probabilities.argmax(axis=1)
mlp = model.named_steps["mlpclassifier"]
print("epochs selected:", mlp.n_iter_)
print("test accuracy:", round(accuracy_score(y_test, predictions), 3))
print("test log loss:", round(log_loss(y_test, probabilities), 3))
```

</details>

This example keeps preprocessing inside a pipeline and leaves the test set untouched until training choices are complete. In a research comparison, hyperparameters should be selected with cross-validation or a dedicated validation protocol rather than by repeatedly inspecting test performance.


### **Representation Learning**

A **representation** is a transformation of raw input into variables that make relevant structure easier to model. Classical feature engineering fixes this transformation before fitting the predictor. Neural representation learning estimates an encoder $h_\phi$ jointly with a task head $g_\psi$:

$$
\mathbf z=h_\phi(\mathbf x),
\qquad
\widehat y=g_\psi(\mathbf z).
$$

$\mathbf z$ may be a hidden vector, a sequence of contextual embeddings, a feature map, or a hierarchy of several resolutions. The final head can remain simple because the encoder changes the geometry of the data.

<div class="diagram-scroll">

![Raw observations passing through a learned encoder into a compact representation used by multiple task heads.](assets/representation-learning.svg){fig-alt="Representation learning maps raw observations to a task-relevant coordinate system that supports simple prediction and transfer."}

</div>

A useful representation often exhibits several properties:

- **task sufficiency:** it retains information needed for the target;
- **invariance:** nuisance changes such as small translations or irrelevant style do not alter the representation too much;
- **equivariance:** when a transformation should change the output predictably, the representation changes in a corresponding way;
- **disentangling or factorization:** distinct explanatory factors become easier to access, although perfect disentanglement is rarely guaranteed;
- **transferability:** features learned from one task or data source remain useful for another.

These goals can conflict. A representation invariant to speaker identity is useful for speech transcription but harmful for speaker verification. The training objective and data augmentation define which distinctions the encoder is encouraged to preserve or discard.

Hidden features are not automatically interpretable. A single neuron need not correspond to one human concept, and the same information can be distributed across many coordinates. Useful diagnostics include:

- **linear probes**, which test whether a target can be recovered with a simple linear head;
- nearest-neighbor inspection in representation space;
- dimensionality-reduction visualizations used cautiously;
- interventions or ablations that test whether features are causally used;
- performance across domains, subgroups, and nuisance transformations.

A high probe score shows that information is linearly accessible, not necessarily that the original model's head uses it or that the representation encodes a causal concept.

<details>
<summary><strong>Python: inspect how a learned hidden layer changes separability</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=1400, noise=0.23, random_state=7)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=7
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline: a linear boundary in the original standardized coordinates.
raw_probe = LogisticRegression().fit(X_train_scaled, y_train)
raw_accuracy = accuracy_score(y_test, raw_probe.predict(X_test_scaled))

# Learn one nonlinear hidden representation.
encoder_model = MLPClassifier(
    hidden_layer_sizes=(12,), activation="relu", alpha=1e-4,
    max_iter=1000, random_state=7
).fit(X_train_scaled, y_train)

W_hidden = encoder_model.coefs_[0]
b_hidden = encoder_model.intercepts_[0]
H_train = np.maximum(0.0, X_train_scaled @ W_hidden + b_hidden)
H_test = np.maximum(0.0, X_test_scaled @ W_hidden + b_hidden)

# Freeze the learned encoder and fit a new linear probe on its features.
hidden_probe = LogisticRegression(max_iter=1000).fit(H_train, y_train)
hidden_accuracy = accuracy_score(y_test, hidden_probe.predict(H_test))

print("raw-feature linear accuracy:", round(raw_accuracy, 3))
print("hidden-feature linear accuracy:", round(hidden_accuracy, 3))
print("representation shape:", H_train.shape)
```

</details>

The hidden probe normally outperforms the raw linear model because the MLP has bent the two-moons geometry into a representation that a hyperplane can separate. This is a small supervised example of a broader principle: neural networks jointly learn a coordinate system and a predictor.

Representation learning becomes especially powerful through **pretraining**. An encoder first learns from a large source task, a self-supervised objective, or multiple tasks. Downstream use can then:

- freeze the encoder and train a new head;
- fine-tune all parameters with a smaller learning rate;
- update only selected layers or parameter-efficient adapters;
- combine learned embeddings with classical models.

Transfer is not guaranteed. Source and target may reward different invariances, pretraining data may contain bias, and full fine-tuning can forget useful features or overfit a small target set. Frozen-head, partial, and full fine-tuning baselines help determine how much adaptation is actually needed.


### **Architecture Map**

An architecture specifies more than layer count. It determines which positions interact, which parameters are shared, and what symmetries or locality assumptions are built into the model. These inductive biases affect sample efficiency, computation, and the kinds of patterns that are easy to learn.

<div class="diagram-scroll">

![MLPs, CNNs, recurrent networks, attention, and Transformers organized by their parameter sharing and context mixing.](assets/neural-architecture-map.svg){fig-alt="A map from input structure to MLP, CNN, recurrent, and attention-based architectural inductive biases."}

</div>

#### **CNNs, RNNs, Attention, and Transformers**

An **MLP** applies dense transformations to a fixed-size vector. It assumes no spatial or temporal structure beyond what features encode. Parameter count grows with input dimension, so an MLP is often used as a head or for compact tabular and embedding inputs.

A **convolutional neural network (CNN)** applies the same local kernel at many positions. In one dimension,

$$
y_t=\sum_{r=0}^{k-1}w_r x_{t+r}+b.
$$

The kernel width $k$ controls the local receptive field, and weight sharing makes the same pattern detector available everywhere. Stacking convolutions expands the receptive field and builds hierarchical spatial features. This bias is well suited to grids and local signals, but long-range interaction may require depth, dilation, pooling, or attention.

<details>
<summary><strong>Python: observe parameter sharing in a one-dimensional convolution</strong></summary>

```python
import numpy as np

signal = np.array([0.0, 1.0, 3.0, 2.0, 2.0, -1.0, 0.0])
kernel = np.array([-1.0, 0.0, 1.0])  # a local change detector

output = []
for start in range(len(signal) - len(kernel) + 1):
    local_window = signal[start:start + len(kernel)]
    output.append(np.dot(local_window, kernel))

print("signal:", signal)
print("shared kernel:", kernel)
print("convolution output:", np.array(output))
print("kernel parameters used at every position:", len(kernel))
```

</details>

A **recurrent neural network (RNN)** reuses a transition function over an ordered sequence:

$$
\mathbf h_t=g(\mathbf W_x\mathbf x_t+\mathbf W_h\mathbf h_{t-1}+\mathbf b).
$$

The hidden state summarizes a prefix and supports streaming computation. Repeated multiplication through time can create vanishing or exploding gradients; LSTM and GRU gates regulate information flow. Sequential recurrence limits parallelism, and a fixed-size state can become an information bottleneck.

**Attention** lets each position form a content-dependent weighted combination of values. Scaled dot-product attention is

$$
\operatorname{Attention}(\mathbf Q,\mathbf K,\mathbf V)
=\operatorname{softmax}\!\left(
\frac{\mathbf Q\mathbf K^\top}{\sqrt{d_k}}
\right)\mathbf V.
$$

Queries determine what each position seeks, keys determine what each source position offers for matching, and values carry the information to combine. Unlike a fixed convolution kernel, attention weights depend on the current input.

<details>
<summary><strong>Python: compute scaled dot-product attention</strong></summary>

```python
import numpy as np

tokens = np.array([
    [1.0, 0.0, 0.5],
    [0.8, 0.2, 0.4],
    [0.0, 1.0, 0.3],
])

# Small illustrative projections; learned models estimate these matrices.
W_query = np.eye(3)
W_key = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.2, 0.2, 1.0]])
W_value = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])

queries = tokens @ W_query
keys = tokens @ W_key
values = tokens @ W_value
scores = queries @ keys.T / np.sqrt(keys.shape[1])
scores -= scores.max(axis=1, keepdims=True)
weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
context = weights @ values

print("attention weights:\n", np.round(weights, 3))
print("row sums:", weights.sum(axis=1))
print("context vectors:\n", np.round(context, 3))
```

</details>

A **Transformer** combines multi-head attention, position information, feedforward sublayers, residual connections, and normalization. Self-attention permits direct pairwise interaction and parallel processing across sequence positions, although standard dense attention has quadratic time and memory in sequence length. Architectural variants change sparsity, recurrence, state caching, or approximation to manage long context.

| Architecture | Main sharing pattern | Strong inductive bias | Typical limitation |
|---|---|---|---|
| MLP | Dense layer weights | Generic fixed-size vector mapping | Parameters ignore locality |
| CNN | Kernel shared across position | Locality and translation-related patterns | Long-range context is indirect |
| RNN / LSTM / GRU | Transition shared across time | Ordered state and streaming | Sequential execution and gradient path length |
| Attention | Content-dependent pairwise weights | Flexible relation lookup | Pairwise cost can be high |
| Transformer | Attention plus position and MLP blocks | Parallel contextual representation | Data, memory, and serving cost |

#### **Connection to the Deep Learning Guideline**

This chapter establishes the common substrate: nonlinear function composition, forward computation, backpropagation, autodiff, optimization, and learned representations. The Deep Learning series should continue from here rather than restart from linear algebra. Its architecture chapters can ask:

- how convolution encodes locality and scale;
- how recurrent gates and state-space mechanisms handle sequence memory;
- how attention, positional information, and Transformers construct context;
- how normalization, residual paths, initialization, and large-scale optimization enable depth;
- how pretraining, transfer, generative objectives, and multimodal systems change the learning problem.

The bridge is therefore conceptual rather than merely chronological. Classical machine learning chooses a hypothesis class and optimization procedure; deep learning does the same, but uses layered representations and architecture-specific parameter sharing at much larger scale.


### **When to Use Classical or Neural Models**

Neural networks are flexible, but flexibility is not a universal advantage. Model choice should account for data modality, sample size, available pretraining, latency, memory, interpretability, uncertainty, maintenance, and the cost of an error.

<div class="diagram-scroll">

![A decision guide that starts from data modality and compares classical baselines, pretrained neural encoders, and hybrid models under shared evaluation.](assets/classical-neural-decision.svg){fig-alt="A practical workflow for choosing classical, neural, or hybrid models using validation and deployment constraints."}

</div>

Classical methods are often strong when:

- data are compact tabular records with meaningful columns;
- the labeled sample is small or medium-sized;
- missingness, monotonicity, sparse effects, or calibrated uncertainty need explicit treatment;
- CPU latency, memory, or explainability is tightly constrained;
- tree ensembles, linear models, or kernels already match the data geometry.

Neural models become especially attractive when:

- inputs are images, text, audio, video, graphs, or other high-dimensional structured signals;
- useful features are difficult to specify manually;
- abundant data or relevant pretrained models are available;
- the system benefits from transfer learning, multitask learning, or end-to-end representation adaptation;
- task quality justifies higher training and serving cost.

Hybrid approaches are common. A pretrained text or image encoder can produce embeddings for a linear model or tree ensemble. A neural network can combine sparse identifiers with dense tabular features. A classical calibrator or decision rule can operate on neural logits. The useful boundary is not ideological; it is an engineering decomposition.

<details>
<summary><strong>Python: compare classical and neural baselines under one protocol</strong></summary>

```python
from sklearn.datasets import make_classification, make_moons
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def compare_models(X, y, name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=12
    )
    models = {
        "logistic": make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=2000)
        ),
        "tree ensemble": HistGradientBoostingClassifier(
            max_iter=150, max_leaf_nodes=15, random_state=12
        ),
        "MLP": make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(32, 16),
                learning_rate_init=3e-3,
                alpha=1e-4,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=30,
                max_iter=1000,
                random_state=12,
            ),
        ),
    }
    print(name)
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        accuracy = accuracy_score(y_test, model.predict(X_test))
        print(f"  {model_name:13s}: {accuracy:.3f}")

X_tabular, y_tabular = make_classification(
    n_samples=1600, n_features=12, n_informative=6,
    n_redundant=2, class_sep=1.2, random_state=12
)
X_moons, y_moons = make_moons(n_samples=1600, noise=0.24, random_state=12)

compare_models(X_tabular, y_tabular, "structured tabular-style data")
compare_models(X_moons, y_moons, "nonlinear geometric data")
```

</details>

The result is dataset-specific and intentionally not a universal leaderboard. A fair comparison tunes each family, reports uncertainty across splits or seeds, includes preprocessing inside the validation loop, and measures more than accuracy. Training time, inference latency, memory, energy, calibration, robustness, and maintenance burden may reverse the preferred choice.

A reliable decision process is:

1. establish a leakage-safe linear or tree baseline;
2. identify whether the main error comes from representation, optimization, insufficient data, label noise, or distribution shift;
3. introduce a neural model only with a clear hypothesis about the expected gain;
4. use pretrained representations before training a large encoder from scratch when appropriate;
5. compare models under the same split, metric, and tuning budget;
6. inspect subgroup, temporal, and out-of-distribution failures;
7. select the smallest system that meets quality and operational requirements.

Neural networks connect the earlier machine-learning toolkit to deep learning because they do not replace its principles. They still require a well-defined target, representative data, an objective, regularization, honest evaluation, and deployment monitoring. Their distinctive contribution is the joint learning of layered representations and predictions through differentiable computation.

The presentation follows the transition from linear models to MLPs in [Dive into Deep Learning](https://classic.d2l.ai/chapter_multilayer-perceptrons/mlp.html), the computational-graph account of backpropagation in [Stanford CS231n](https://cs231n.github.io/optimization-2/), and the activation and hidden-layer treatment in [Google's Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course/neural-networks). The XOR bridge is also consistent with the introductory neural-network treatment in [CMU's machine-learning notes](https://www.cs.cmu.edu/~15281-s24/lectures/ML_notes_S24.pdf). The diagrams are stored locally as original teaching illustrations so the rendered blog does not depend on fragile external image URLs.
